# Hidden Carriers — Reference solution (ORGANISERS ONLY — do not publish)

**Story.** An infectious disease (SIR) spreads through a community made of households, schools and workplaces
(5 communities per town). On observation day, about **40 % of the infected people are asymptomatic**: contact tracers
see nothing (`observed == 0`) although they are infectious.

**Your task (node level).** For every person who *looks healthy* (`observed == 0`) predict the probability that they are
**already infected or will be infected within the next 3 days**. A good ranking = a good quarantine priority list.

| `observed` | meaning |
|---|---|
| 0 | looks healthy (susceptible **or** hidden/asymptomatic carrier) → **these rows are scored** |
| 1 | visibly infected (symptomatic) |
| 2 | recovered |

**Files** (`train_*` has labels, `test_*` does not; each *graph* is one independent town / outbreak)
* `*_nodes.csv`: `graph_id, node_id, observed, n_contacts, contact_intensity, community` (+ `target` in train, only for `observed==0`)
* `*_edges.csv`: `graph_id, src, dst, weight` — undirected contacts, stored once (`src < dst`), `weight ∈ [0.2, 1]` = contact intensity
* `sample_submission.csv`: `id = "<graph_id>_<node_id>", target = probability`

**Data is messy on purpose:** `contact_intensity` and `community` are missing for ~5 % of people (incomplete contact tracing).

**Metric:** ROC-AUC over all scored test rows (public LB = 18 test graphs, private LB = the other 42).

**Rules of the game**
* Validate **by graph** (`GroupKFold(groups=graph_id)`), never by node — neighbours leak labels across a node split.
* 0-hop (own features) is blind, 1-hop (visible neighbours) is decent, **2+ hops (hidden carriers) is where the signal is**.
* Accuracy is meaningless here (~12 % positives). Use AUC.

**What this notebook contains:** three ways to clean the data, hand-crafted graph features, a Random Forest, a
HistGradientBoosting model, a PyTorch GCN trained in **batches of graphs**, and a rank-blend → `submission.csv`.

## 1. Hyper-parameters

In [ ]:
HP = dict(
    # ---------- data cleaning ----------
    clean_method      = 'edges',   # "median" | "group_median" | "edges"   -> compare them with CV below!
    add_missing_flags = True,   # keep the 2 "was this value missing?" indicator columns? True / False

    # ---------- RandomForest (sklearn) ----------
    rf_n_estimators   = 500,   # e.g. 200-1000
    rf_min_samples_leaf = 10,   # e.g. 1-50  (bigger = smoother; labels are noisy!)
    rf_max_features   = 0.5,   # e.g. 0.2-1.0 (fraction of features per split)

    # ---------- HistGradientBoosting (sklearn) ----------
    gb_max_iter       = 300,   # e.g. 100-600
    gb_learning_rate  = 0.05,   # e.g. 0.02-0.2
    gb_max_leaf_nodes = 15,   # e.g. 4-63
    gb_l2             = 1.0,   # L2 regularisation, e.g. 0-10

    # ---------- 2-layer GCN (PyTorch, batches of graphs) ----------
    gcn_hidden        = 32,   # hidden width, e.g. 16-128
    gcn_dropout       = 0.2,   # 0.0-0.5
    gcn_lr            = 0.01,   # Adam learning rate, e.g. 0.003-0.03
    gcn_weight_decay  = 0.0005,   # e.g. 0-1e-3
    gcn_epochs        = 80,   # full passes over the training graphs
    gcn_batch_graphs  = 4,   # graphs per optimiser step (1 = pure SGD over graphs)
    gcn_pos_weight    = 'auto',   # "auto" (= #neg/#pos) or a float; 1.0 = no re-weighting
    gcn_n_seeds       = 3,   # how many GCNs to average

    # ---------- blending (weights of rank-averaged predictions; 0 = ignore a model) ----------
    blend_weights     = {'rf': 0.2, 'gb': 0.4, 'gcn': 0.4},   # dict(rf=?, gb=?, gcn=?)
)
missing = [k for k, val in HP.items() if val is None]
assert not missing, f"Fill in these hyper-parameters first: {missing}"
SEED = 0
print("hyper-parameters OK")

## 2. Setup and data

In [ ]:
import os, glob, time, copy, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
np.random.seed(SEED)
N = 500   # people per town

def find_data_dir():
    hits = glob.glob("/kaggle/input/*/train_nodes.csv") + glob.glob("/kaggle/input/*/*/train_nodes.csv") + glob.glob("./train_nodes.csv") + glob.glob("../input/*/train_nodes.csv")
    assert hits, "train_nodes.csv not found - attach the competition data to the notebook"
    return os.path.dirname(hits[0])
DATA = find_data_dir()
train_nodes = pd.read_csv(f"{DATA}/train_nodes.csv"); train_edges = pd.read_csv(f"{DATA}/train_edges.csv")
test_nodes  = pd.read_csv(f"{DATA}/test_nodes.csv");  test_edges  = pd.read_csv(f"{DATA}/test_edges.csv")
sample_sub  = pd.read_csv(f"{DATA}/sample_submission.csv")
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print(DATA, "| train graphs:", train_nodes.graph_id.nunique(), "| test graphs:", test_nodes.graph_id.nunique())
train_nodes.head()

In [ ]:
cand = train_nodes[train_nodes.observed == 0]
print(f"scored rows in train: {len(cand):,}   positive rate: {cand.target.mean():.1%}   (accuracy is a trap!)")
print("missing values:\n", train_nodes[["observed", "n_contacts", "contact_intensity", "community"]].isna().mean().round(3).to_string())
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
train_nodes.observed.value_counts().sort_index().plot.bar(ax=ax[0], title="observed state (0 = looks healthy)")
cand.groupby("graph_id").target.mean().plot.hist(bins=15, ax=ax[1], title="positive rate per graph")
train_nodes.n_contacts.plot.hist(bins=25, ax=ax[2], title="contacts per person"); plt.tight_layout(); plt.show()

## 3. Data cleaning — three strategies
* `median` — global median / mode (the classic).
* `group_median` — median within people with the same number of contacts.
* `edges` — **recompute** the intensity exactly from the edge list and **vote** the community from the neighbours' communities.

Which one is best is an empirical question — compare them with graph-grouped CV in section 5.

In [ ]:
def graph_arrays(nodes_g, edges_g):
    """Dense weighted adjacency W (N x N) and node table sorted by node_id for ONE graph."""
    nd = nodes_g.sort_values("node_id").reset_index(drop=True)
    W = np.zeros((N, N), dtype=np.float32)
    W[edges_g.src.to_numpy(), edges_g.dst.to_numpy()] = edges_g.weight.to_numpy()
    W = W + W.T
    return nd, W

def clean_graph(nd, W, method="edges"):
    """Fill the two tracing gaps. method: 'median' | 'group_median' | 'edges' (exact recompute / neighbour vote)."""
    nd = nd.copy()
    nd["miss_intensity"] = nd.contact_intensity.isna().astype(int)
    nd["miss_community"] = nd.community.isna().astype(int)
    if method == "median":
        nd["contact_intensity"] = nd.contact_intensity.fillna(nd.contact_intensity.median())
        nd["community"] = nd.community.fillna(nd.community.mode().iloc[0])
    elif method == "group_median":
        nd["contact_intensity"] = nd.contact_intensity.fillna(nd.groupby("n_contacts").contact_intensity.transform("median"))
        nd["contact_intensity"] = nd.contact_intensity.fillna(nd.contact_intensity.median())
        nd["community"] = nd.community.fillna(nd.community.mode().iloc[0])
    elif method == "edges":
        nd["contact_intensity"] = nd.contact_intensity.fillna(pd.Series(W.sum(1)))
        com = nd.community.to_numpy(dtype=float).copy()
        known = ~np.isnan(com)
        onehot = np.zeros((N, 5), dtype=np.float32); onehot[np.where(known)[0], com[known].astype(int)] = 1
        votes = (W > 0).astype(np.float32) @ onehot                      # neighbour vote (communities are dense)
        guess = np.where(votes.sum(1) > 0, votes.argmax(1), np.nan)
        com[~known] = guess[~known]
        nd["community"] = pd.Series(com).fillna(pd.Series(com).mode().iloc[0])
    else:
        raise ValueError(method)
    nd["community"] = nd["community"].astype(int)
    return nd

## 4. Graph features: 0-hop, 1-hop, 2-hop, community- and town-level

In [ ]:
def node_features(nd, W):
    """Hand-crafted 0/1/2-hop + community + graph-level features for ONE graph."""
    obs = nd.observed.to_numpy(); A = (W > 0).astype(np.float32)
    vis = (obs == 1).astype(np.float32); rec = (obs == 2).astype(np.float32); cand = (obs == 0).astype(np.float32)
    com = nd.community.to_numpy(); C = np.eye(5, dtype=np.float32)[com]
    deg = np.maximum(A.sum(1), 1)
    f = pd.DataFrame({
        "n_contacts": nd.n_contacts.to_numpy(), "contact_intensity": nd.contact_intensity.to_numpy(),
        "miss_intensity": nd.miss_intensity.to_numpy(), "miss_community": nd.miss_community.to_numpy(),
        "h1_vis_w": W @ vis, "h1_vis_n": A @ vis, "h1_rec_w": W @ rec, "h1_rec_n": A @ rec,
        "h1_vis_frac": (A @ vis) / deg, "h1_cand_n": A @ cand,
        "h2_vis_w": W @ (W @ vis), "h2_vis_n": A @ (A @ vis),
        "h2_via_cand": W @ (cand * (W @ vis)),                           # visible -> (maybe hidden) -> me
        "h2_via_cand_n": A @ (cand * (A @ vis)),
        "nbr_mean_h1": (A @ (W @ vis)) / deg,
        "com_vis_frac": ((C * vis[:, None]).sum(0) / C.sum(0).clip(1))[com],
        "com_rec_frac": ((C * rec[:, None]).sum(0) / C.sum(0).clip(1))[com],
        "g_vis_frac": np.full(N, vis.mean()), "g_rec_frac": np.full(N, rec.mean()),
    })
    for k in range(5): f[f"com_{k}"] = C[:, k]
    return f

def build_table(nodes, edges, method="edges", with_target=True):
    """Candidate-node feature table for every graph in (nodes, edges)."""
    rows, eg = [], {g: d for g, d in edges.groupby("graph_id")}
    for gid, ng in nodes.groupby("graph_id"):
        nd, W = graph_arrays(ng, eg[gid])
        nd = clean_graph(nd, W, method); f = node_features(nd, W)
        f.insert(0, "graph_id", gid); f.insert(1, "node_id", nd.node_id.to_numpy()); f["observed"] = nd.observed.to_numpy()
        if with_target: f["target"] = nd.target.to_numpy()
        rows.append(f[f.observed == 0])
    return pd.concat(rows, ignore_index=True)

In [ ]:
t0 = time.time()
TR = build_table(train_nodes, train_edges, HP["clean_method"])
TE = build_table(test_nodes, test_edges, HP["clean_method"], with_target=False)
drop = ["graph_id", "node_id", "observed", "target"] + ([] if HP["add_missing_flags"] else ["miss_intensity", "miss_community"])
FEATURES = [c for c in TR.columns if c not in drop]
y, groups = TR.target.astype(int), TR.graph_id
print(f"{len(FEATURES)} features, {len(TR):,} train rows, {len(TE):,} test rows  ({time.time()-t0:.0f}s)")
print("bar to beat on train - 1-hop heuristic AUC: %.3f" % roc_auc_score(y, TR.h1_vis_w))

## 5. Models and graph-grouped cross-validation

In [ ]:
def make_rf():
    return RandomForestClassifier(n_estimators=HP["rf_n_estimators"], min_samples_leaf=HP["rf_min_samples_leaf"],
                                  max_features=HP["rf_max_features"], n_jobs=-1, random_state=SEED)
def make_gb():
    return HistGradientBoostingClassifier(max_iter=HP["gb_max_iter"], learning_rate=HP["gb_learning_rate"],
                                          max_leaf_nodes=HP["gb_max_leaf_nodes"], l2_regularization=HP["gb_l2"], random_state=SEED)

def cv_pooled_auc(make_model, X, y, groups, n_splits=5):
    """Out-of-fold AUC pooled over all rows (same flavour as the leaderboard); folds split BY GRAPH."""
    oof = np.zeros(len(y))
    for tr, va in GroupKFold(n_splits).split(X, y, groups):
        oof[va] = make_model().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[va])[:, 1]
    return roc_auc_score(y, oof), oof

cv_rf, oof_rf = cv_pooled_auc(make_rf, TR[FEATURES], y, groups); print(f"RandomForest        CV AUC = {cv_rf:.4f}")
cv_gb, oof_gb = cv_pooled_auc(make_gb, TR[FEATURES], y, groups); print(f"HistGradientBoosting CV AUC = {cv_gb:.4f}")

In [ ]:
# OPTIONAL: which cleaning strategy is best?  (rebuilds the feature table 3 times, ~1 min)
RUN_CLEANING_COMPARISON = False
if RUN_CLEANING_COMPARISON:
    for method in ["median", "group_median", "edges"]:
        T = build_table(train_nodes, train_edges, method)
        cols = [c for c in T.columns if c not in drop]
        auc, _ = cv_pooled_auc(make_gb, T[cols], T.target.astype(int), T.graph_id)
        print(f"clean_method={method:13s} GB CV AUC = {auc:.4f}")

In [ ]:
t0 = time.time()
rf, gb = make_rf().fit(TR[FEATURES], y), make_gb().fit(TR[FEATURES], y)
TE["rf"], TE["gb"] = rf.predict_proba(TE[FEATURES])[:, 1], gb.predict_proba(TE[FEATURES])[:, 1]
imp = pd.Series(rf.feature_importances_, FEATURES).sort_values().tail(12); imp.plot.barh(title="RandomForest: top features", figsize=(6, 3.5)); plt.show()
print(f"fitted on all train graphs ({time.time()-t0:.0f}s)")

## 6. PyTorch GCN, trained in batches of graphs
`H' = σ(Â H W)` with `Â = D^-1/2 (A + I) D^-1/2` (**self-loops!**, weighted edges). Two layers = two hops. One *graph* is one full-graph sample; an optimiser step uses `gcn_batch_graphs` graphs (gradient accumulation).

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(SEED)

def normalize_adj(W):
    A = torch.tensor(W) + torch.eye(W.shape[0])            # A~ = A + I  (a node must keep its own features)
    d_inv_sqrt = A.sum(1).pow(-0.5)
    return d_inv_sqrt[:, None] * A * d_inv_sqrt[None, :]

def make_gcn_graphs(nodes, edges, with_target, stats=None):
    """One dict per graph: x [N,F], A_hat [N,N], mask (observed==0), y, ids. Uses the chosen cleaning method."""
    eg = {g: d for g, d in edges.groupby("graph_id")}
    raw = []
    for gid, ng in nodes.groupby("graph_id"):
        nd, W = graph_arrays(ng, eg[gid]); nd = clean_graph(nd, W, HP["clean_method"])
        raw.append((gid, nd, W))
    if stats is None:
        ci = np.concatenate([nd.contact_intensity.to_numpy() for _, nd, _ in raw]); stats = (ci.mean(), ci.std() + 1e-6)
    graphs = []
    for gid, nd, W in raw:
        cols = [np.eye(3, dtype=np.float32)[nd.observed.to_numpy()],
                ((nd.contact_intensity.to_numpy() - stats[0]) / stats[1])[:, None].astype(np.float32),
                np.eye(5, dtype=np.float32)[nd.community.to_numpy()]]
        if HP["add_missing_flags"]:
            cols.append(nd[["miss_intensity", "miss_community"]].to_numpy(dtype=np.float32))
        graphs.append(dict(gid=gid, ids=nd.node_id.to_numpy(), x=torch.tensor(np.concatenate(cols, axis=1)), A_hat=normalize_adj(W),
                           mask=torch.tensor(nd.observed.to_numpy() == 0),
                           y=torch.tensor(nd.target.fillna(0).to_numpy(dtype=np.float32)) if with_target else None))
    return graphs, stats

t0 = time.time()
gcn_train_all, stats = make_gcn_graphs(train_nodes, train_edges, True)
gcn_test, _ = make_gcn_graphs(test_nodes, test_edges, False, stats)
print(f"{len(gcn_train_all)} train / {len(gcn_test)} test graph tensors, {gcn_train_all[0]['x'].shape[1]} input features ({time.time()-t0:.0f}s)")

In [ ]:
class GCNLayer(nn.Module):
    """H' = A_hat @ H @ W + b   (activation applied outside)."""
    def __init__(self, i, o):
        super().__init__(); self.W = nn.Linear(i, o, bias=False); self.b = nn.Parameter(torch.zeros(o))
    def forward(self, A_hat, H):
        return A_hat @ self.W(H) + self.b

class GCN(nn.Module):
    """2 layers -> 2-hop receptive field; returns one logit per node."""
    def __init__(self, in_dim, hidden, dropout):
        super().__init__(); self.l1, self.l2, self.dropout = GCNLayer(in_dim, hidden), GCNLayer(hidden, 1), dropout
    def forward(self, x, A_hat):
        h = F.dropout(F.relu(self.l1(A_hat, x)), self.dropout, self.training)
        return self.l2(A_hat, h).squeeze(-1)

def graph_loss(model, g, pos_w, n_nodes):
    logits = model(g["x"], g["A_hat"]); m = g["mask"]
    return F.binary_cross_entropy_with_logits(logits[m], g["y"][m], pos_weight=pos_w, reduction="sum") / n_nodes

def train_gcn(train_g, val_g, seed):
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    n_pos = sum(g["y"][g["mask"]].sum().item() for g in train_g); n_neg = sum((1 - g["y"][g["mask"]]).sum().item() for g in train_g)
    pw = n_neg / n_pos if HP["gcn_pos_weight"] == "auto" else float(HP["gcn_pos_weight"])
    pos_w = torch.tensor([pw], dtype=torch.float32)
    model = GCN(train_g[0]["x"].shape[1], HP["gcn_hidden"], HP["gcn_dropout"])
    opt = torch.optim.Adam(model.parameters(), lr=HP["gcn_lr"], weight_decay=HP["gcn_weight_decay"])
    B, best, best_state, hist = HP["gcn_batch_graphs"], float("inf"), None, []
    for ep in range(HP["gcn_epochs"]):
        model.train(); order = rng.permutation(len(train_g))
        for b in range(0, len(order), B):
            batch = [train_g[i] for i in order[b:b + B]]
            n_nodes = sum(int(g["mask"].sum()) for g in batch)
            opt.zero_grad()                                         # 1. zero_grad
            for g in batch:                                         # 2-4. forward + loss + backward, accumulated over the batch
                graph_loss(model, g, pos_w, n_nodes).backward()
            opt.step()                                              # 5. step
        model.eval()
        with torch.no_grad():
            val = float(np.mean([graph_loss(model, g, pos_w, int(g["mask"].sum())).item() for g in val_g]))
        hist.append(val)
        if val < best: best, best_state = val, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, hist

@torch.no_grad()
def predict_gcn(model, graphs):
    model.eval(); rows = []
    for g in graphs:
        s = model(g["x"], g["A_hat"]).numpy(); m = g["mask"].numpy()
        rows.append(pd.DataFrame({"graph_id": g["gid"], "node_id": g["ids"][m], "gcn": s[m]}))
    return pd.concat(rows, ignore_index=True)

In [ ]:
# hold out some TRAIN GRAPHS for early stopping (split by graph!)
perm = np.random.RandomState(SEED).permutation(len(gcn_train_all))
val_g = [gcn_train_all[i] for i in perm[:15]]; train_g = [gcn_train_all[i] for i in perm[15:]]
t0, preds, curves = time.time(), [], []
for s in range(HP["gcn_n_seeds"]):
    model, hist = train_gcn(train_g, val_g, seed=SEED + s)
    preds.append(predict_gcn(model, gcn_test)); curves.append(hist)
    print(f"GCN seed {s}: best val loss {min(hist):.4f}  ({time.time()-t0:.0f}s)")
gcn_pred = preds[0].copy(); gcn_pred["gcn"] = np.mean([p["gcn"].to_numpy() for p in preds], axis=0)    # logits are averaged
for h in curves: plt.plot(h)
plt.title("GCN validation loss (held-out train graphs)"); plt.xlabel("epoch"); plt.show()
TE = TE.merge(gcn_pred, on=["graph_id", "node_id"], how="left"); assert TE.gcn.notna().all()

## 7. Blend and write `submission.csv`

In [ ]:
def rank01(v): return rankdata(v) / len(v)
w = HP["blend_weights"]; assert sum(w.values()) > 0
TE["blend"] = (w["rf"] * rank01(TE.rf) + w["gb"] * rank01(TE.gb) + w["gcn"] * rank01(TE.gcn)) / sum(w.values())
sub = pd.DataFrame({"id": TE.graph_id.astype(str) + "_" + TE.node_id.astype(str), "target": TE.blend})
sub = sample_sub[["id"]].merge(sub, on="id", how="left")           # same rows, same order as the sample
assert len(sub) == len(sample_sub) and sub.target.notna().all() and np.isfinite(sub.target).all() and not sub.id.duplicated().any()
sub.to_csv(f"{OUT}/submission.csv", index=False)
print(f"wrote {OUT}/submission.csv  ({len(sub):,} rows)"); sub.head()

## 8. Ideas to climb the leaderboard
* Compare the three cleaning strategies with `RUN_CLEANING_COMPARISON = True`; try dropping/keeping the missing-flags.
* More hops: build `W @ (cand * (W @ (cand * (W @ vis))))` (visible → hidden → hidden → me), random-walk / PageRank scores from the visible cases.
* Town-level features matter for a *pooled* AUC (how far along is this outbreak? `g_vis_frac`, `g_rec_frac`).
* GCN: try residual / skip connections, a 3rd layer, different adjacency normalisations, degree-aware features.
* Seeds + blending: average several seeds; tune `blend_weights` on out-of-fold predictions, not on the leaderboard.
* The leaderboard is noisy (±0.005–0.01 between close solutions). Trust graph-grouped CV, not one public score.